In [3]:
from utils import load_encoder_hparams_and_params
encoder, hparams, params = load_encoder_hparams_and_params("124M", "models")

Fetching checkpoint: 1.00kb [00:00, 2.24Mb/s]                                                       
Fetching encoder.json: 1.04Mb [00:00, 2.57Mb/s]                                                     
Fetching hparams.json: 1.00kb [00:00, 10.0Mb/s]                                                     
Fetching model.ckpt.data-00000-of-00001: 498Mb [04:05, 2.02Mb/s]                                    
Fetching model.ckpt.index: 6.00kb [00:00, 8.71Mb/s]                                                 
Fetching model.ckpt.meta: 472kb [00:00, 1.95Mb/s]                                                   
Fetching vocab.bpe: 457kb [00:00, 2.14Mb/s]                                                         


In [8]:
import numpy as np
from pprint import pprint
def shape_tree(d):
     if isinstance(d, np.ndarray):
         return list(d.shape)
     elif isinstance(d, list):
         return [shape_tree(v) for v in d]
     elif isinstance(d, dict):
         return {k: shape_tree(v) for k, v in d.items()}
     else:
         ValueError("uh oh")
         
pprint(shape_tree(params))

{'blocks': [{'attn': {'c_attn': {'b': [2304], 'w': [768, 2304]},
                      'c_proj': {'b': [768], 'w': [768, 768]}},
             'ln_1': {'b': [768], 'g': [768]},
             'ln_2': {'b': [768], 'g': [768]},
             'mlp': {'c_fc': {'b': [3072], 'w': [768, 3072]},
                     'c_proj': {'b': [768], 'w': [3072, 768]}}},
            {'attn': {'c_attn': {'b': [2304], 'w': [768, 2304]},
                      'c_proj': {'b': [768], 'w': [768, 768]}},
             'ln_1': {'b': [768], 'g': [768]},
             'ln_2': {'b': [768], 'g': [768]},
             'mlp': {'c_fc': {'b': [3072], 'w': [768, 3072]},
                     'c_proj': {'b': [768], 'w': [3072, 768]}}},
            {'attn': {'c_attn': {'b': [2304], 'w': [768, 2304]},
                      'c_proj': {'b': [768], 'w': [768, 768]}},
             'ln_1': {'b': [768], 'g': [768]},
             'ln_2': {'b': [768], 'g': [768]},
             'mlp': {'c_fc': {'b': [3072], 'w': [768, 3072]},
               

In [11]:
def gelu(x):
    return 0.5 * x * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

def layer_norm(x, g, b, eps: float = 1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    x = (x - mean) / np.sqrt(variance + eps)  # normalize x to have mean=0 and var=1 over last axis
    return g * x + b  # scale and offset with gamma/beta params

# Linear layers are often referred to as projections (since they are projecting from one vector space to 
# another vector space).
def linear(x, w, b):  # [m, in], [in, out], [out] -> [m, out]
    return x @ w + b
print(softmax([1, 2, 3]))
print(layer_norm([1, 2, 4], 1, 0))

[0.09003057 0.24472847 0.66524096]
[-1.06904153 -0.26726038  1.33630191]


In [ ]:
def position_feed_forward_network(x, c_fc, c_proj):  # [n_seq, n_embd] -> [n_seq, n_embd]
    # project up
    a = gelu(linear(x, **c_fc))  # [n_seq, n_embd] -> [n_seq, 4*n_embd]

    # project back down
    x = linear(a, **c_proj)  # [n_seq, 4*n_embd] -> [n_seq, n_embd]

    return x


def attention(q, k, v):  # [n_q, d_k], [n_k, d_k], [n_k, d_v] -> [n_q, d_v]
    return softmax(q @ k.T / np.sqrt(q.shape[-1])) @ v

# We can enhance self attention by introducing projections for q, k, v and the attention output.
# This enables our model to learn a mapping for q, k, and v that best helps attention distinguish 
# relationships between inputs.
def self_attention(x, w_k, w_q, w_v, w_proj): # [n_seq, n_embd] -> [n_seq, n_embd]
    # qkv projections
    q = x @ w_k # [n_seq, n_embd] @ [n_embd, n_embd] -> [n_seq, n_embd]
    k = x @ w_q # [n_seq, n_embd] @ [n_embd, n_embd] -> [n_seq, n_embd]
    v = x @ w_v # [n_seq, n_embd] @ [n_embd, n_embd] -> [n_seq, n_embd]

    # perform self attention
    x = attention(q, k, v) # [n_seq, n_embd] -> [n_seq, n_embd]

    # out projection
    x = x @ w_proj # [n_seq, n_embd] @ [n_embd, n_embd] -> [n_seq, n_embd]

    return x

# Causal - means hide the future tokens
# We can reduce the number of matrix multiplication from 4 to just 2 
# if we combine w_q, w_k and w_v into a single matrix w_fc, perform the projection, and then split the result
def causal_self_attention(x, c_attn, c_proj): # [n_seq, n_embd] -> [n_seq, n_embd]
    # qkv projections
    x = linear(x, **c_attn) # [n_seq, n_embd] -> [n_seq, 3*n_embd]

    # split into qkv
    q, k, v = np.split(x, 3, axis=-1) # [n_seq, 3*n_embd] -> 3 of [n_seq, n_embd]

    # causal mask to hide future inputs from being attended to
    causal_mask = (1 - np.tri(x.shape[0]), dtype=x.dtype) * -1e10  # [n_seq, n_seq]

    # perform causal self attention
    x = attention(q, k, v, causal_mask) # [n_seq, n_embd] -> [n_seq, n_embd]

    # out projection
    x = linear(x, **c_proj) # [n_seq, n_embd] @ [n_embd, n_embd] = [n_seq, n_embd]

    return x

def multi_headed_causal_self_attention(x, c_attn, c_proj, n_head):  # [n_seq, n_embd] -> [n_seq, n_embd]
    # qkv projection
    x = linear(x, **c_attn)  # [n_seq, n_embd] -> [n_seq, 3*n_embd]

    # split into qkv
    qkv = np.split(x, 3, axis=-1)  # [n_seq, 3*n_embd] -> [3, n_seq, n_embd]

    # split into heads
    qkv_heads = list(map(lambda x: np.split(x, n_head, axis=-1), qkv))  # [3, n_seq, n_embd] -> [3, n_head, n_seq, n_embd/n_head]

    # causal mask to hide future inputs from being attended to
    causal_mask = (1 - np.tri(x.shape[0], dtype=x.dtype)) * -1e10  # [n_seq, n_seq]

    # perform attention over each head
    out_heads = [attention(q, k, v, causal_mask) for q, k, v in zip(*qkv_heads)]  # [3, n_head, n_seq, n_embd/n_head] -> [n_head, n_seq, n_embd/n_head]

    # merge heads
    x = np.hstack(out_heads)  # [n_head, n_seq, n_embd/n_head] -> [n_seq, n_embd]

    # out projection
    x = linear(x, **c_proj)  # [n_seq, n_embd] -> [n_seq, n_embd]

    return x

In [ ]:


def transformer_block(x, mlp, attn, ln_1, ln_2, n_head):  # [n_seq, n_embd] -> [n_seq, n_embd]
    # multi-head causal self attention
    x = x + multi_headed_causal_self_attention(layer_norm(x, **ln_1), **attn, n_head=n_head)  # [n_seq, n_embd] -> [n_seq, n_embd]

    # position-wise feed forward network
    x = x + position_feed_forward_network(layer_norm(x, **ln_2), **mlp)  # [n_seq, n_embd] -> [n_seq, n_embd]

    return x

def gpt_overview(inputs, wte, wpe, blocks, ln_f, n_head):  # [n_seq] -> [n_seq, n_vocab]
    # token + positional embeddings
    x = wte[inputs] + wpe[range(len(inputs))]  # [n_seq] -> [n_seq, n_embd]

    # forward pass through n_layer transformer blocks
    for block in blocks:
        x = transformer_block(x, **block, n_head=n_head)  # [n_seq, n_embd] -> [n_seq, n_embd]

    # projection to vocab
    x = layer_norm(x, **ln_f)  # [n_seq, n_embd] -> [n_seq, n_embd]
    return x @ wte.T  # [n_seq, n_embd] -> [n_seq, n_vocab]